In [ ]:
import os
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader

from dataset import CheXpertDataset, CHEXPERT_PATHOLOGY_LABELS, get_transforms
from models import get_model
from vlg_cbm_lib.datasets import ConceptLayer, BackboneWithConcepts
from utils.saliency import (
    GradCAM, 
    IntegratedGradients, 
    SmoothGrad,
    ConceptAttributionMap,
    visualize_heatmap,
    save_saliency_visualization
)

%matplotlib inline
plt.style.use('default')

## Configuration

In [ ]:
# Paths
MODEL_DIR = "checkpoints/vlg_cbm_10k"
DATA_DIR = "/workspace/CheXpert-v1.0-small"
CONCEPTS_FILE = "concepts/chexpert_concepts.txt"
OUTPUT_DIR = "visualizations/interactive"

# Settings
BACKBONE = "densenet121"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IMG_SIZE = 224

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Using device: {DEVICE}")

## Load Model and Data

In [ ]:
# Load concepts
with open(CONCEPTS_FILE) as f:
    concept_names = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(concept_names)} concepts")
print("First 5 concepts:")
for i, c in enumerate(concept_names[:5]):
    print(f"  {i}: {c}")

In [ ]:
# Load model configuration
config_path = os.path.join(MODEL_DIR, "config.json")
with open(config_path) as f:
    config = json.load(f)

n_concepts = config["n_concepts"]
feature_dim = config.get("feature_dim", 1024)

print(f"Model config: {n_concepts} concepts, {feature_dim}D features")

In [ ]:
# Build model
backbone_model = get_model(BACKBONE, num_classes=12, pretrained=False)
backbone = backbone_model.backbone.features

concept_layer = ConceptLayer(
    input_dim=feature_dim,
    n_concepts=n_concepts,
    num_hidden=config.get("num_hidden", 1),
    hidden_dim=config.get("hidden_dim")
)

model = BackboneWithConcepts(backbone, concept_layer)

# Load weights
ckpt_path = os.path.join(MODEL_DIR, "best_model.pth")
state_dict = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(state_dict, strict=False)
model.to(DEVICE)
model.eval()

# Load final layer weights
W_path = os.path.join(MODEL_DIR, "W_c.pt")
W = torch.load(W_path, map_location=DEVICE, weights_only=False)

print("Model loaded successfully!")
print(f"Final layer shape: {W.shape}")

In [ ]:
# Load dataset
transform = get_transforms(train=False, img_size=IMG_SIZE)
csv_path = os.path.join(DATA_DIR, "valid.csv")

dataset = CheXpertDataset(
    csv_path=csv_path,
    transform=transform,
    uncertain_strategy="ones",
    use_frontal_only=True,
    label_subset=CHEXPERT_PATHOLOGY_LABELS
)

dataloader = DataLoader(dataset, batch_size=1, shuffle=True)
print(f"Loaded {len(dataset)} validation samples")

## 1. Single Concept Visualization

In [ ]:
# Get a sample image
image, label = next(iter(dataloader))
image = image.to(DEVICE)

print(f"Image shape: {image.shape}")
print(f"Label: {label}")

In [ ]:
# Compute concept activations
with torch.no_grad():
    concept_logits = model(image)
    concept_scores = torch.sigmoid(concept_logits).squeeze().cpu().numpy()

# Show top activated concepts
top_k = 10
top_indices = np.argsort(concept_scores)[-top_k:][::-1]

print(f"\nTop {top_k} activated concepts:")
for rank, idx in enumerate(top_indices, 1):
    print(f"{rank}. [{idx}] {concept_names[idx]}: {concept_scores[idx]:.3f}")

In [ ]:
# Visualize saliency for a specific concept
CONCEPT_IDX = top_indices[0]  # Most activated concept

# Initialize GradCAM
gradcam = GradCAM(model.backbone)

# Generate heatmap
heatmap = gradcam.generate_cam(image, model.concept_layer, CONCEPT_IDX)

# Resize to image size
import cv2
heatmap_resized = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))

print(f"\nGenerated GradCAM for concept: {concept_names[CONCEPT_IDX]}")
print(f"Activation score: {concept_scores[CONCEPT_IDX]:.3f}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original image
img_np = image.squeeze().cpu().numpy()
if img_np.shape[0] == 3:
    img_np = np.transpose(img_np, (1, 2, 0))
img_display = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)

axes[0].imshow(img_display, cmap='gray' if img_np.ndim == 2 else None)
axes[0].set_title("Original Image")
axes[0].axis('off')

# Heatmap
im = axes[1].imshow(heatmap_resized, cmap='jet', vmin=0, vmax=1)
axes[1].set_title("Saliency Map")
axes[1].axis('off')
plt.colorbar(im, ax=axes[1], fraction=0.046)

# Overlay
overlay = visualize_heatmap(image, heatmap_resized, alpha=0.5)
axes[2].imshow(overlay)
axes[2].set_title("Overlay")
axes[2].axis('off')

fig.suptitle(f"Concept: {concept_names[CONCEPT_IDX]}\nScore: {concept_scores[CONCEPT_IDX]:.3f}",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 2. Compare Different Saliency Methods

In [ ]:
# Generate saliency maps with different methods
CONCEPT_IDX = top_indices[0]

# GradCAM
gradcam = GradCAM(model.backbone)
heatmap_gradcam = gradcam.generate_cam(image, model.concept_layer, CONCEPT_IDX)
heatmap_gradcam = cv2.resize(heatmap_gradcam, (IMG_SIZE, IMG_SIZE))

# Integrated Gradients
ig = IntegratedGradients(model)
heatmap_ig = ig.generate_attribution(image, CONCEPT_IDX, n_steps=30)

# SmoothGrad
sg = SmoothGrad(model, noise_level=0.15, n_samples=20)
heatmap_sg = sg.generate_attribution(image, CONCEPT_IDX)

print("Generated saliency maps with all methods")

In [ ]:
# Compare visualizations
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Heatmaps
methods = ['GradCAM', 'Integrated Gradients', 'SmoothGrad']
heatmaps = [heatmap_gradcam, heatmap_ig, heatmap_sg]

for i, (method, hmap) in enumerate(zip(methods, heatmaps)):
    im = axes[0, i].imshow(hmap, cmap='jet', vmin=0, vmax=1)
    axes[0, i].set_title(f"{method}")
    axes[0, i].axis('off')
    plt.colorbar(im, ax=axes[0, i], fraction=0.046)

# Row 2: Overlays
for i, (method, hmap) in enumerate(zip(methods, heatmaps)):
    overlay = visualize_heatmap(image, hmap, alpha=0.5)
    axes[1, i].imshow(overlay)
    axes[1, i].set_title(f"{method} Overlay")
    axes[1, i].axis('off')

fig.suptitle(f"Concept: {concept_names[CONCEPT_IDX]}\nComparison of Saliency Methods",
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Disease-Level Attribution Through Concepts

In [ ]:
# Create disease attribution visualizer
disease_attr = ConceptAttributionMap(
    backbone=model.backbone,
    concept_layer=model.concept_layer,
    final_layer_weights=W
)

print("Disease attribution initialized")
print(f"Diseases: {CHEXPERT_PATHOLOGY_LABELS}")

In [ ]:
# Select a disease to analyze
DISEASE_IDX = 0  # e.g., "Enlarged Cardiomediastinum"
DISEASE_NAME = CHEXPERT_PATHOLOGY_LABELS[DISEASE_IDX]

# Generate attribution
top_concept_indices, weights, combined_heatmap = disease_attr.generate_disease_attribution(
    image,
    disease_idx=DISEASE_IDX,
    top_k=10
)

print(f"\nTop concepts for {DISEASE_NAME}:")
for idx, weight in zip(top_concept_indices, weights):
    print(f"  {concept_names[idx][:60]}: {weight:.4f}")

In [ ]:
# Visualize disease attribution
fig = plt.figure(figsize=(18, 6))

# Original image
ax1 = plt.subplot(1, 3, 1)
img_np = image.squeeze().cpu().numpy()
if img_np.shape[0] == 3:
    img_np = np.transpose(img_np, (1, 2, 0))
img_display = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
ax1.imshow(img_display, cmap='gray' if img_np.ndim == 2 else None)
ax1.set_title("Original Image")
ax1.axis('off')

# Attribution heatmap overlay
ax2 = plt.subplot(1, 3, 2)
overlay = visualize_heatmap(image, combined_heatmap, alpha=0.5)
ax2.imshow(overlay)
ax2.set_title(f"Attribution for: {DISEASE_NAME}")
ax2.axis('off')

# Top concepts bar chart
ax3 = plt.subplot(1, 3, 3)
concept_labels = [concept_names[i][:50] + '...' if len(concept_names[i]) > 50 
                 else concept_names[i] for i in top_concept_indices]
colors = ['red' if w < 0 else 'green' for w in weights]
ax3.barh(range(len(weights)), weights, color=colors, alpha=0.7)
ax3.set_yticks(range(len(weights)))
ax3.set_yticklabels(concept_labels, fontsize=9)
ax3.set_xlabel("Concept Weight", fontsize=11)
ax3.set_title("Top Contributing Concepts", fontsize=11)
ax3.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
ax3.grid(axis='x', alpha=0.3)
ax3.invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Batch Analysis: Top Concepts Across Multiple Images

In [ ]:
# Analyze concept activations across multiple samples
n_samples = 50
all_concept_scores = []

for i, (img, lbl) in enumerate(dataloader):
    if i >= n_samples:
        break
    img = img.to(DEVICE)
    with torch.no_grad():
        scores = torch.sigmoid(model(img)).squeeze().cpu().numpy()
    all_concept_scores.append(scores)

all_concept_scores = np.array(all_concept_scores)
mean_scores = all_concept_scores.mean(axis=0)
std_scores = all_concept_scores.std(axis=0)

print(f"Computed concept statistics over {n_samples} samples")

In [ ]:
# Plot most activated concepts on average
top_k = 15
top_mean_indices = np.argsort(mean_scores)[-top_k:][::-1]

fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(top_mean_indices))
means = mean_scores[top_mean_indices]
stds = std_scores[top_mean_indices]

ax.bar(x_pos, means, yerr=stds, capsize=5, alpha=0.7, color='steelblue')
ax.set_xticks(x_pos)
labels = [concept_names[i][:40] + '...' if len(concept_names[i]) > 40 
          else concept_names[i] for i in top_mean_indices]
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Mean Activation Score', fontsize=11)
ax.set_title(f'Top {top_k} Concepts by Average Activation\n(over {n_samples} samples)', 
             fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Interactive Exploration

In [ ]:
# Function to visualize any concept for any image
def explore_concept(sample_idx=0, concept_idx=0, method='gradcam'):
    """
    Interactively explore concept saliency.
    
    Args:
        sample_idx: Index of sample in dataset
        concept_idx: Index of concept to visualize
        method: 'gradcam', 'integrated_gradients', or 'smoothgrad'
    """
    # Get sample
    image, label = dataset[sample_idx]
    image = image.unsqueeze(0).to(DEVICE)
    
    # Get concept activation
    with torch.no_grad():
        concept_logits = model(image)
        concept_scores = torch.sigmoid(concept_logits).squeeze().cpu().numpy()
    
    # Generate saliency
    if method == 'gradcam':
        gradcam = GradCAM(model.backbone)
        heatmap = gradcam.generate_cam(image, model.concept_layer, concept_idx)
        heatmap = cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))
    elif method == 'integrated_gradients':
        ig = IntegratedGradients(model)
        heatmap = ig.generate_attribution(image, concept_idx)
    elif method == 'smoothgrad':
        sg = SmoothGrad(model)
        heatmap = sg.generate_attribution(image, concept_idx)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    img_np = image.squeeze().cpu().numpy()
    if img_np.shape[0] == 3:
        img_np = np.transpose(img_np, (1, 2, 0))
    img_display = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
    
    axes[0].imshow(img_display, cmap='gray' if img_np.ndim == 2 else None)
    axes[0].set_title("Original")
    axes[0].axis('off')
    
    im = axes[1].imshow(heatmap, cmap='jet', vmin=0, vmax=1)
    axes[1].set_title("Saliency")
    axes[1].axis('off')
    plt.colorbar(im, ax=axes[1], fraction=0.046)
    
    overlay = visualize_heatmap(image, heatmap, alpha=0.5)
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay")
    axes[2].axis('off')
    
    concept_name = concept_names[concept_idx]
    score = concept_scores[concept_idx]
    fig.suptitle(f"Sample {sample_idx} | Concept: {concept_name}\nScore: {score:.3f} | Method: {method}",
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return image, heatmap

print("Function 'explore_concept' defined. Usage:")
print("  explore_concept(sample_idx=5, concept_idx=10, method='gradcam')")

In [ ]:
# Try it out!
# Change these values to explore different concepts and images
img, hmap = explore_concept(sample_idx=0, concept_idx=5, method='gradcam')

## Summary

This notebook demonstrated:

1. **GradCAM**: Shows spatial regions that activate concepts
2. **Integrated Gradients**: Path-based attribution with theoretical guarantees
3. **SmoothGrad**: Noise-averaged gradients for cleaner visualizations
4. **Disease Attribution**: Understanding predictions through interpretable concepts

These tools enable:
- Validating that concepts capture meaningful visual patterns
- Debugging model behavior
- Building trust through interpretability
- Clinical validation of model decisions